In [1]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType
spark = SparkSession.builder \
    .appName("word2vec_token_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [2]:
#file parquet nằm trong iceberg-warehouse
df_tokens = spark.read.table("my_catalog.processed_zone_finnhub.news_tokens")
df_tokens.show(5, truncate=False)

+---------+-------------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id       |published_at       |title_tokens                                                                                       |summary_tokens                                                                                                                                            

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Word2Vec, StringIndexer
from pyspark.ml.classification import  RandomForestClassifier  # Hoặc RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import concat, col

# 2. Nạp dữ liệu từ bảng Iceberg đã lưu ở bước trước
iceberg_table_path = "my_catalog.processed_zone_finnhub.news_tokens_sentiment_labeled"
print(f"[*] Đang nạp dữ liệu từ kho Iceberg: {iceberg_table_path}")
df_raw = spark.read.table(iceberg_table_path)

# Kiểm tra dữ liệu đầu vào
print("[*] Cấu trúc dữ liệu vừa nạp:")
df_raw.select("id", "title_tokens", "summary_polarity_sentiment").show(5, truncate=40)

df_combined = df_raw.withColumn(
    "combined_tokens", 
    concat(col("title_tokens"), col("summary_tokens"))
)


# 3. Định nghĩa Pipeline
# CẬP NHẬT QUAN TRỌNG: Đổi inputCol của Word2Vec sang cột mới gộp
word2vec = Word2Vec(
    vectorSize=100,
    minCount=3,
    windowSize=5,
    inputCol="combined_tokens", # <--- SỬA Ở ĐÂY
    outputCol="w2v_features",
    maxIter=20
)



# Bước 3.2: Chuyển đổi nhãn cảm xúc dạng chữ thành mã số (Ví dụ: Tích cực -> 0.0, Tiêu cực -> 1.0...)
label_indexer = StringIndexer(
    inputCol="summary_polarity_sentiment", 
    outputCol="indexed_label",
    handleInvalid="keep"  # Tránh lỗi nếu xuất hiện nhãn lạ
)

# Bước 3.3: Khởi tạo thuật toán phân loại (Classifier)
# Ở đây sử dụng Logistic Regression (Hồi quy Logistic đa lớp), rất mạnh và nhanh với dữ liệu text vector
classifier = RandomForestClassifier(featuresCol="w2v_features", labelCol="indexed_label", numTrees=50, maxDepth=10)

# 4. Gom tất cả các bước vào một Pipeline duy nhất
# Điểm hay của Pipeline là khi dự đoán dữ liệu mới, text thô đi vào sẽ tự động biến thành số rồi ra kết quả luôn
pipeline = Pipeline(stages=[word2vec, label_indexer, classifier])


# 5. Chia tập dữ liệu huấn luyện và kiểm thử (80% Train / 20% Test)
print("[*] Đang phân tách dữ liệu thành tập Train (80%) và Test (20%)...")
train_data, test_data = df_combined.randomSplit([0.8, 0.2], seed=42)

import time

# 6. Tiến hành Huấn luyện toàn bộ hệ thống
print("[*] Đang khởi chạy quy trình huấn luyện toàn diện (Word2Vec + Classifier)...")

# Bắt đầu bấm giờ
start_time = time.time()

# Chạy lệnh huấn luyện
pipeline_model = pipeline.fit(train_data)

# Kết thúc bấm giờ và tính toán
end_time = time.time()
training_time_seconds = end_time - start_time
training_time_minutes = training_time_seconds / 60

# In ra kết quả thời gian cụ thể
print(f"[+] Huấn luyện mô hình thành công!")
print(f"    ⏱️ Thời gian chạy: {training_time_seconds:.2f} giây (tương đương khoảng {training_time_minutes:.2f} phút).")


# 7. Đánh giá chất lượng mô hình trên tập Test độc lập
print("[*] Đang chạy thử nghiệm dự đoán trên tập dữ liệu Test...")
predictions = pipeline_model.transform(test_data)

# Khởi tạo bộ đo đa lớp để tính toán độ chính xác tổng quan
evaluator = MulticlassClassificationEvaluator(
    labelCol="indexed_label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)

print(f"\n=======================================================")
print(f"🔥 ĐỘ CHÍNH XÁC TỔNG QUAN CỦA MÔ HÌNH (ACCURACY): {accuracy * 100:.2f}%")
print(f"=======================================================")

# Hiển thị thực tế kết quả dự đoán đối chiếu với nhãn gốc
print("\n[*] Bảng kết quả dự đoán mẫu trên tập Test:")
predictions.select(
    "title_tokens", 
    "summary_polarity_sentiment", 
    "indexed_label", 
    "prediction"
).show(20, truncate=40)


# 8. Lưu trữ Pipeline Model hoàn chỉnh lên Amazon S3
# Lưu dạng Pipeline giúp bạn lưu cả từ điển Word2Vec lẫn bộ phân loại vào một thư mục chung
model_save_path = "s3a://raw-financial-data/models/news_sentiment_pipeline_model_tokens"
print(f"[*] Đang đóng gói và lưu trữ mô hình tại: {model_save_path}")
pipeline_model.write().overwrite().save(model_save_path)

print("[+] Quy trình hoàn tất! Mô hình đã sẵn sàng để đưa vào sử dụng thực tế.")

[*] Đang nạp dữ liệu từ kho Iceberg: my_catalog.processed_zone_finnhub.news_tokens_sentiment_labeled
[*] Cấu trúc dữ liệu vừa nạp:
+---------+----------------------------------------+--------------------------+
|       id|                            title_tokens|summary_polarity_sentiment|
+---------+----------------------------------------+--------------------------+
|140651233|[market, chatter, jpmorgan, google, f...|              Trung tính ⚪|
|140651234|[uk, regulator, sets, new, rules, goo...|               Tích cực 🟢|
|140650770|[google, cloud, model, garden, platfo...|              Trung tính ⚪|
|140650768|[social, media, political, problem, t...|              Trung tính ⚪|
|140649980|[custom, ai, chips, coming, nvidia, c...|               Tích cực 🟢|
+---------+----------------------------------------+--------------------------+
only showing top 5 rows

[*] Đang phân tách dữ liệu thành tập Train (80%) và Test (20%)...
[*] Đang khởi chạy quy trình huấn luyện toàn diện (Word2Vec 

In [7]:
# Hiển thị thực tế kết quả dự đoán đối chiếu với nhãn gốc
print("\n[*] Bảng kết quả dự đoán mẫu trên tập Test:")
predictions.select(
    "title_tokens", 
    "summary_polarity_sentiment", 
    "indexed_label", 
    "prediction"
).show(20, truncate=40)



[*] Bảng kết quả dự đoán mẫu trên tập Test:
+----------------------------------------+--------------------------+-------------+----------+
|                            title_tokens|summary_polarity_sentiment|indexed_label|prediction|
+----------------------------------------+--------------------------+-------------+----------+
|              [best, stocks, buy, watch]|               Tích cực 🟢|          1.0|       1.0|
|              [best, stocks, buy, watch]|               Tích cực 🟢|          1.0|       1.0|
|[wall, street, bulls, look, optimisti...|              Trung tính ⚪|          0.0|       1.0|
|              [best, stocks, buy, watch]|               Tích cực 🟢|          1.0|       1.0|
|[ai, stocks, face, moment, phase, ai,...|               Tích cực 🟢|          1.0|       0.0|
|[stocks, analyst, favorites, magnific...|               Tích cực 🟢|          1.0|       0.0|
|[crwd, panw, sentinelone, set, secure...|               Tích cực 🟢|          1.0|       0.0|
|[boeing, d